# 02 — Data Cleaning

Apply imputation, outlier capping, and implausible-value removal.
Report quality improvement before vs after.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.cleaning import impute_missing, cap_outliers, remove_implausible, final_quality, initial_quality

In [ ]:
df_raw = pd.read_csv("../data/raw/okcupid_profiles.csv")
print(f"Raw: {len(df_raw):,} rows")

## Step 1 — Impute missing values
- Numeric (height): median
- Income: recode `-1` sentinel to NaN
- Categorical: 'unknown' sentinel string
- Essays: empty string

In [ ]:
df_imputed = impute_missing(df_raw)
print("After imputation:")
print(df_imputed.isna().mean().sort_values(ascending=False).head(10).to_string())

## Step 2 — Cap outliers (Winsorize at 0.5%/99.5%)

In [ ]:
df_capped = cap_outliers(df_imputed)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes, ['age', 'height', 'income'], ['#4A90E2', '#7ED321', '#F5A623']):
    df_raw[col].dropna().plot.box(ax=ax, vert=False, positions=[1], widths=0.5, patch_artist=True,
                                   boxprops=dict(facecolor='#FF6B6B', alpha=0.5), label='before')
    df_capped[col].dropna().plot.box(ax=ax, vert=False, positions=[2], widths=0.5, patch_artist=True,
                                      boxprops=dict(facecolor=color, alpha=0.7), label='after')
    ax.set_title(col)
    ax.set_yticks([1, 2])
    ax.set_yticklabels(['before', 'after'])
plt.tight_layout()
plt.show()

## Step 3 — Remove implausible rows

In [ ]:
df_clean = remove_implausible(df_capped)
print(f"Final: {len(df_clean):,} rows ({len(df_clean) / len(df_raw):.1%} kept)")

## Quality improvement summary

In [ ]:
summary = final_quality(df_raw, df_clean)
for k, v in summary.items():
    if isinstance(v, float) and v < 1:
        print(f"  {k:30s} {v:>10.1%}")
    else:
        print(f"  {k:30s} {v:>10}")

## Persist cleaned dataset

In [ ]:
processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)
df_clean.to_parquet(processed_dir / 'okcupid_clean.parquet', index=False)
print(f"Saved to {processed_dir / 'okcupid_clean.parquet'}")